In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-03 04:32:34.079025: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-03 04:32:34.785742: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [1]:
import os

def clean():
    folder_paths = ["logs"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
    "learning_type": "DL",
    "lib": "tensorflow",
    "mode": "local",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions


In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-03 04:32:35,797 [DEBUG] [Rain] Rain is initialized
2023-07-03 04:32:35,799 [DEBUG] [Provisioner] Creating coordinator
2023-07-03 04:32:35,800 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-03 04:32:35,801 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-03 04:32:35,802 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized


Creating temporary directory ../../../../Rain/data/coord/data/
Creating temporary directory ../../../../Rain/data/coord/data/
Creating temporary directory ../../../../Rain/data/divider/data/
Creating temporary directory ../../../../Rain/data/divider/data/
Creating temporary directory ../../../../Rain/data/divider/data/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-03 04:32:35,809 [DEBUG] [Rain] Creating workers
2023-07-03 04:32:35,814 [INFO] [Provisioner] provisioner is serving
2023-07-03 04:32:35,815 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 04:32:35,817 [INFO] [Coordinator] coordinator is serving
2023-07-03 04:32:35,817 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 04:32:35,821 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 04:32:35,822 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 04:32:35,823 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 04:32:35,826 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 04:32:35,828 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-03 04:32:35,830 [INFO] [Worker_50153] Worker is running on port: 50153
2023-07-03 04:32:35,831 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0

Creating temporary directory ../../../../Rain/data/worker/data/
Creating temporary directory ../../../../Rain/data/worker/data/
Creating temporary directory ../../../../Rain/data/worker/data/


2023-07-03 04:32:40,058 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 04:32:40,115 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 04:32:44,287 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 04:32:44,343 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 04:32:48,400 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 04:32:48,456 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 04:32:48,457 [DEBUG] [DividerProxy] Training Started
2023-07-03 04:32:48,518 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-03 04:32:48,519 [DEBUG] [Provisioner] Received '' from the coordinator to send status
2023-07-03 04:32:48,520 [DEBUG] [Provisioner] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152,

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.7149 - accuracy: 0.7728
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.7069 - accuracy: 0.7774
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.7112 - accuracy: 0.7754
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.3192 - accuracy: 0.9031
sending data to coordinator
157/157 [==============================] - 1s 4ms/step - loss: 0.3122 - accuracy: 0.9064
sending data to coordinator


2023-07-03 04:33:08,033 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 04:33:08,033 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 04:33:08,086 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 04:33:08,087 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 04:33:08,162 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1 successfully
2023-07-03 04:33:08,180 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-03 04:33:08,182 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 04:33:08,183 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl fro

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 7ms/step - loss: 0.2575 - accuracy: 0.9222
Epoch 2/2
157/157 [==============================] - 1s 7ms/step - loss: 0.2148 - accuracy: 0.9342
sending data to coordinator
157/157 [==============================] - 1s 7ms/step - loss: 0.1999 - accuracy: 0.9395
sending data to coordinator


2023-07-03 04:33:20,993 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 04:33:20,994 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 04:33:21,015 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 04:33:21,017 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 04:33:21,093 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 04:33:21,094 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 04:33:21,168 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1 successfully
2023-07-03 04:33:21,178 [DEBUG] [DeepLearning] Asynchronous update is done by

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 6ms/step - loss: 0.1903 - accuracy: 0.9425
Epoch 2/2
157/157 [==============================] - 1s 6ms/step - loss: 0.2341 - accuracy: 0.9294
Epoch 2/2
157/157 [==============================] - 1s 6ms/step - loss: 0.2073 - accuracy: 0.9376
Epoch 2/2
157/157 [==============================] - 1s 6ms/step - loss: 0.1595 - accuracy: 0.9513
sending data to coordinator
157/157 [==============================] - 1s 6ms/step - loss: 0.1865 - accuracy: 0.9435
sending data to coordinator
157/157 [==============================] - 1s 6ms/step - loss: 0.1758 - accuracy: 0.9479
sending data to coordinator


2023-07-03 04:33:27,689 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 04:33:27,691 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 04:33:27,705 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 04:33:27,706 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 04:33:27,714 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 04:33:27,715 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 04:33:27,912 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2 successfully
2023-07-03 04:33:27,922 [DEBUG] [DeepLearning] Asynchronous update is done by

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1084 - accuracy: 0.9688

Test accuracy: 96.9%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-03 04:33:28,357 [DEBUG] [Rain] Creating workers
2023-07-03 04:33:28,359 [INFO] [Provisioner] provisioner is serving
2023-07-03 04:33:28,359 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 04:33:28,361 [INFO] [Coordinator] coordinator is serving
2023-07-03 04:33:28,361 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 04:33:28,363 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 04:33:28,364 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 04:33:28,365 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 04:33:28,368 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 04:33:28,368 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 04:33:28,371 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-03 04:33:28,371 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-03 04:33:28,374 [I

Creating temporary directory ../../../../Rain/data/worker/data/
Creating temporary directory ../../../../Rain/data/worker/data/
Creating temporary directory ../../../../Rain/data/worker/data/


2023-07-03 04:33:32,737 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 04:33:32,801 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 04:33:37,520 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 04:33:37,578 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 04:33:43,761 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 04:33:43,819 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 04:33:43,820 [DEBUG] [DividerProxy] Training Started
2023-07-03 04:33:43,822 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-03 04:33:43,823 [DEBUG] [Provisioner] Received '' from the coordinator to send status
2023-07-03 04:33:43,824 [DEBUG] [Provisioner] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152,

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 7ms/step - loss: 0.1576 - accuracy: 0.9520
Epoch 2/2
157/157 [==============================] - 2s 7ms/step - loss: 0.1621 - accuracy: 0.9516
Epoch 2/2
157/157 [==============================] - 1s 7ms/step - loss: 0.1289 - accuracy: 0.9612
sending data to coordinator
157/157 [==============================] - 1s 7ms/step - loss: 0.1358 - accuracy: 0.9597
sending data to coordinator
157/157 [==============================] - 1s 7ms/step - loss: 0.1400 - accuracy: 0.9585
sending data to coordinator


2023-07-03 04:34:04,888 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 04:34:04,889 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_1_trained.pkl from worker3
2023-07-03 04:34:04,897 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 04:34:04,898 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 04:34:04,922 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 04:34:04,924 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_1_trained.pkl from worker2
2023-07-03 04:34:05,152 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_1_trained.pkl from worker3 successfully
2023-07-03 04:34:05,163 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1303 - accuracy: 0.9602
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1416 - accuracy: 0.9569
Epoch 2/2
157/157 [==============================] - 1s 9ms/step - loss: 0.1182 - accuracy: 0.9637
sending data to coordinator
sending data to coordinator
157/157 [==============================] - 1s 8ms/step - loss: 0.1209 - accuracy: 0.9628
sending data to coordinator


2023-07-03 04:34:12,739 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 04:34:12,740 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 04:34:12,742 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_2_trained.pkl from worker3
2023-07-03 04:34:12,742 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 04:34:12,901 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 04:34:12,902 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_2_trained.pkl from worker1
2023-07-03 04:34:12,986 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_2_trained.pkl from worker3 successfully
2023-07-03 04:34:12,987 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 7ms/step - loss: 0.1214 - accuracy: 0.9633
Epoch 2/2
157/157 [==============================] - 2s 7ms/step - loss: 0.1128 - accuracy: 0.9646
Epoch 2/2
157/157 [==============================] - 2s 7ms/step - loss: 0.1233 - accuracy: 0.9640
Epoch 2/2
157/157 [==============================] - 1s 7ms/step - loss: 0.1067 - accuracy: 0.9661
sending data to coordinator
sending data to coordinator
sending data to coordinator


2023-07-03 04:34:20,371 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 04:34:20,372 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_3_trained.pkl from worker2
2023-07-03 04:34:20,459 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_3_trained.pkl from worker2 successfully
2023-07-03 04:34:20,557 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 04:34:20,558 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 04:34:20,590 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 04:34:20,591 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_3_trained.pkl from worker1
2023-07-03 04:34:20,688 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0766 - accuracy: 0.9776

Test accuracy: 97.8%
